# Panel de Longevidad — Interpretación de Biomarcadores
**Laboratorio de Análisis Clínicos · Área de Longevidad**

Modos disponibles:
- `scoring` — Clasificación determinística con rangos Medicina 3.0. No requiere API.
- `llm` — Scoring + narrativa clínica integrada vía Claude (requiere `ANTHROPIC_API_KEY`).

In [ ]:
# ══════════════════════════════════════════════════════
#  CONFIGURACIÓN — modificar antes de correr
# ══════════════════════════════════════════════════════

CSV_PATH      = "muestra_panel.csv"   # ruta al archivo exportado del LIS
MODO          = "scoring"             # "scoring" | "llm"
PACIENTE_ID   = None                  # None = todos | "P001" = solo ese paciente
OUTPUT_DIR    = "./informes"          # carpeta de salida para los PDFs

# Solo necesario si MODO = "llm"
# Podés poner la clave acá (solo para uso local) o en el archivo .env
ANTHROPIC_API_KEY = ""  # dejar vacío si usás .env

In [ ]:
import sys
import os
from pathlib import Path

# Asegurar que el directorio del notebook esté en el path
nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').parent.resolve()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

# Cargar .env si existe
env_path = nb_dir / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if "=" in line and not line.strip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

# Clave desde celda de config (prioridad sobre .env)
if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"Directorio: {nb_dir}")
print(f"Modo: {MODO.upper()}")
print(f"API key cargada: {'Sí' if os.environ.get('ANTHROPIC_API_KEY') else 'No (solo scoring)'}")

In [ ]:
import csv
import json
import pandas as pd
from scoring_engine import procesar_paciente

# ── Leer CSV ──
csv_full = nb_dir / CSV_PATH if not Path(CSV_PATH).is_absolute() else Path(CSV_PATH)

if not csv_full.exists():
    print(f"[ERROR] No se encontró: {csv_full}")
    print("Revisá que CSV_PATH apunte al archivo correcto.")
    raise FileNotFoundError(csv_full)

with open(csv_full, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

if PACIENTE_ID:
    rows = [r for r in rows if r.get("paciente_id") == PACIENTE_ID]
    if not rows:
        raise ValueError(f"Paciente '{PACIENTE_ID}' no encontrado en el CSV")

print(f"Pacientes cargados: {len(rows)}")

# Preview
df_preview = pd.read_csv(csv_full, encoding="utf-8-sig")
if PACIENTE_ID:
    df_preview = df_preview[df_preview["paciente_id"] == PACIENTE_ID]
print(f"\nColumnas disponibles ({len(df_preview.columns)}):")
print(", ".join(df_preview.columns.tolist()))
df_preview[[c for c in ["paciente_id","fecha","edad","sexo"] if c in df_preview.columns]].head()

In [ ]:
from scoring_engine import SCORE_MAP, OPTIMO, BUENO, ATENCION, RIESGO, CRITICO

COLORES_NIVEL = {
    OPTIMO:   "background-color: #EAF3DE; color: #27500A",
    BUENO:    "background-color: #E1F5EE; color: #085041",
    ATENCION: "background-color: #FAEEDA; color: #854F0B",
    RIESGO:   "background-color: #FAECE7; color: #993C1D",
    CRITICO:  "background-color: #FCEBEB; color: #A32D2D; font-weight: bold",
}

def color_nivel(val):
    return COLORES_NIVEL.get(str(val), "")

# ── Procesar todos los pacientes ──
resultados = []
for row in rows:
    panel = procesar_paciente(row)
    resultados.append(panel)
    print(f"[{panel.paciente_id}] Score: {panel.score_global} ({panel.score_global_numerico:.2f}) | "
          f"Categorías: {len(panel.categorias)} | Patrones: {len(panel.patrones)} | "
          f"Alertas: {len(panel.alertas)}")

print(f"\nProcesados: {len(resultados)} pacientes")

In [ ]:
# ── Resumen visual por paciente ──
# Si hay múltiples pacientes, mostrá uno a la vez cambiando el índice
IDX = 0  # índice del paciente a mostrar (0 = primero)

panel = resultados[IDX]
print(f"{'═'*55}")
print(f"  Paciente: {panel.paciente_id} | Edad: {panel.edad} | Sexo: {panel.sexo}")
print(f"  Fecha: {panel.fecha}")
print(f"  Score global: {panel.score_global} ({panel.score_global_numerico:.2f})")
print(f"  Edad biológica estimada: {panel.edad_biologica_estimada}")
print(f"{'═'*55}")

# Tabla de categorías
filas = []
for cat in panel.categorias:
    peor_bio = max(
        (b for b in cat.biomarcadores if b.clasificacion and b.valor is not None),
        key=lambda b: SCORE_MAP.get(b.clasificacion, 0),
        default=None
    )
    alerta_cat = any(b.es_alerta for b in cat.biomarcadores)
    filas.append({
        "Categoría": cat.nombre_display,
        "Score": cat.score,
        "Biomarcadores": len([b for b in cat.biomarcadores if b.valor is not None]),
        "Peor resultado": f"{peor_bio.nombre} = {peor_bio.valor} {peor_bio.unidad}" if peor_bio else "—",
        "Alerta": "⚠️" if alerta_cat else "",
    })

df_cats = pd.DataFrame(filas)
df_cats.style.applymap(color_nivel, subset=["Score"])

In [ ]:
# ── Detalle por categoría ──
# Cambiar CATEGORIA para ver otra (ej: "lipidos", "hepatico", "hormonas")
CATEGORIA = "inflamacion"

cat_obj = next((c for c in panel.categorias if c.nombre_key == CATEGORIA), None)
if cat_obj is None:
    print(f"Categoría '{CATEGORIA}' no disponible en este panel.")
    print("Categorías disponibles:", [c.nombre_key for c in panel.categorias])
else:
    print(f"\n{cat_obj.nombre_display} — Score: {cat_obj.score} ({cat_obj.score_numerico:.2f})\n")
    detalle = []
    for b in cat_obj.biomarcadores:
        delta_str = f"{'+' if b.delta and b.delta > 0 else ''}{b.delta}" if b.delta is not None else "—"
        detalle.append({
            "Biomarcador": b.nombre + (" (calc.)" if b.es_calculado else ""),
            "Valor": f"{b.valor} {b.unidad}" if b.valor is not None else "—",
            "Clasificación": b.clasificacion or "—",
            "Rango M3.0": b.rango_longevidad,
            "Delta": f"{delta_str} ({b.tendencia})" if b.tendencia else "—",
            "Alerta": b.nota_alerta if b.es_alerta else "",
        })
    df_det = pd.DataFrame(detalle)
    df_det.style.applymap(color_nivel, subset=["Clasificación"])

In [ ]:
# ── Patrones inter-sistema detectados ──
if not panel.patrones:
    print("No se detectaron patrones inter-sistema en este panel.")
else:
    for p in panel.patrones:
        color = "\033[91m" if p["relevancia"] == "Alta" else "\033[93m"
        reset = "\033[0m"
        print(f"{color}▶ {p['patron']} [{p['relevancia']}]{reset}")
        print(f"  Biomarcadores: {', '.join(p['biomarcadores'])}")
        print(f"  {p['descripcion']}")
        print()

In [ ]:
# ── Alertas críticas ──
if not panel.alertas:
    print("Sin alertas críticas en este panel.")
else:
    print(f"ALERTAS ({len(panel.alertas)}):")
    for a in panel.alertas:
        print(f"  ⚠  {a.nombre}: {a.valor} {a.unidad} — {a.nota_alerta}")

In [ ]:
# ══════════════════════════════════════════════════════
#  MODO LLM — Interpretación clínica vía Claude
#  Solo se ejecuta si MODO = "llm" y hay API key
# ══════════════════════════════════════════════════════

llm_results = {}  # paciente_id -> dict

if MODO == "llm":
    if not os.environ.get("ANTHROPIC_API_KEY"):
        print("[ERROR] ANTHROPIC_API_KEY no configurada. Agregala en la celda de configuración o en el archivo .env")
    else:
        from main import call_llm
        rows_map = {r["paciente_id"]: r for r in rows}

        for panel in resultados:
            pid = panel.paciente_id
            print(f"[{pid}] Llamando Claude...")
            try:
                row_data = rows_map.get(pid, {})
                llm_result = call_llm(panel, row_data)
                llm_results[pid] = llm_result
                print(f"  OK — hallazgos: {len(llm_result.get('hallazgos_por_categoria', []))}")
            except Exception as e:
                print(f"  [ERROR] {e}")
                llm_results[pid] = None
else:
    print("Modo scoring — sin llamada al LLM. Cambiá MODO a 'llm' para activar la interpretación clínica.")

In [ ]:
# ── Mostrar narrativa clínica (si hay resultado LLM) ──
if MODO == "llm" and llm_results:
    pid = resultados[IDX].paciente_id
    res = llm_results.get(pid)
    if res:
        print(f"RESUMEN EJECUTIVO:\n{res.get('resumen_ejecutivo', '—')}\n")
        print(f"NARRATIVA CLÍNICA:\n{res.get('narrativa_clinica', '—')}\n")
        print("PROTOCOLO DIRe — PRIORIDADES:")
        for pr in res.get("protocolo_dire", {}).get("prioridades", []):
            print(f"\n  Prioridad {pr['orden']}: {pr['objetivo']}")
            for iv in pr.get("intervenciones", []):
                nota = f" [{iv['nota_arg']}]" if iv.get("nota_arg") else ""
                print(f"    - [{iv['tipo']}] {iv['descripcion']} (evidencia: {iv['evidencia']}){nota}")
    else:
        print("No hay resultado LLM para este paciente.")

In [ ]:
# ══════════════════════════════════════════════════════
#  GENERAR PDFs
# ══════════════════════════════════════════════════════

from report_generator import generate_pdf
from datetime import datetime

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

pdfs_generados = []
rows_map = {r["paciente_id"]: r for r in rows}

for panel in resultados:
    pid = panel.paciente_id
    llm_res = llm_results.get(pid) if MODO == "llm" else None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = str(out_dir / f"informe_{pid}_{MODO}_{ts}.pdf")
    generate_pdf(panel, out_path, MODO, llm_res)
    pdfs_generados.append(out_path)
    print(f"  PDF: {out_path}")

print(f"\nTotal generados: {len(pdfs_generados)}")